# Workflow Simulation for AI-Assisted Technical Support

This notebook simulates the operational workflow of an AI-assisted technical support system. The workflow integrates knowledge-base retrieval, confidence-based response handling, and escalation logic to model realistic customer support operations.

The objective is to evaluate how AI-driven support systems can automate issue handling while escalating uncertain or unresolved customer queries to human support teams.

In [70]:
import pandas as pd 
import numpy as np

In [71]:
df=pd.read_csv(r"C:\Users\VAIBHAV\model_ready_kb.csv")
df.head()

,flags,instruction,category,intent,response,clean_instruction
0,BLZ,I can't talk with a human agent,CONTACT,contact_human_agent,Thank you for reaching out! I'm sorry to hear ...,i cant talk with a human agent
1,BLMZ,I have got to locate hte bills from {{Person N...,INVOICE,check_invoice,I'll get right on it! I understand your need t...,i have got to locate hte bills from person name
2,BCELM,"I cannot pay, help me to inform of a problem w...",PAYMENT,payment_issue,I apologize for the inconvenience caused by th...,i cannot pay help me to inform of a problem wi...
3,BL,I want help speaking to customer service,CONTACT,contact_customer_service,Grateful for your contact! I get the sense tha...,i want help speaking to customer service
4,BLZ,I try to see th accepted payment options,PAYMENT,check_payment_methods,Thank you for reaching out! I completely under...,i try to see th accepted payment options


## TF-IDF vectorization

In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json
import re

In [73]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(
    df['clean_instruction']
)
tfidf_matrix.shape

(1000, 545)

In [74]:
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_df=0.85,
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(df['clean_instruction'])
custom_stopwords = ['get', 'want', 'need', 'help', 'question']
    
def process_query(query):
    query = query.lower()
    query = re.sub(r'[^a-zA-Z0-9\s]', '', query)
    
    words = query.split()
    words = [word for word in words if word not in custom_stopwords]
    query = ' '.join(words)

    login_clues = ['into my account', 'access account', 'access my account', 'sign in', 'signin', 'login', 'log in', 'cant access', 'cannot access']
    
    if any(clue in query for clue in login_clues):
        query += ' login signin access password registration problems'

    return query

## Workflow Simulation Execution

This section simulates the operational workflow of the AI-assisted support system by processing customer queries, retrieving the most relevant knowledge-base response, evaluating confidence scores, and determining whether the issue should be resolved automatically or escalated to human support.

In [75]:
process_query('How do I get into my account')

'how do i into my account login signin access password registration problems'

In [76]:
workflow_logs=[]

In [79]:
user_query = input("Enter a new query: ")

clean_query = process_query(user_query)

print("Clean Query:", clean_query)

query_vector = tfidf.transform([clean_query])

similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
).flatten()
best_match_index = similarity_scores.argmax()
best_match = df.iloc[best_match_index]
best_score = similarity_scores[best_match_index]

if best_score >= 0.70:
    decision = 'AI Resolved'
    print(decision)
    print("Category:", best_match['category'])
    print("Intent:", best_match['intent'])
    print("Response:", best_match['response'])
    print("Similarity Score:", best_score)
elif best_score >= 0.55:
    decision = 'AI Suggested Response - Confirmation Recommended'
    print(decision)
    print("Category:", best_match['category'])
    print("Intent:", best_match['intent'])
    print("Suggested Response:", best_match['response'])
    print("Similarity Score:", best_score)
else:
    decision = 'Escalated to Human Support'
    print(decision)
    print("Similarity Score:", best_score)
workflow_log = {
    "original_query": user_query,
    "processed_query": clean_query,
    "matched_category": best_match['category'],
    "matched_intent": best_match['intent'],
    "similarity_score": best_score,
    "decision":decision 
}

Enter a new query:  I need refund


Clean Query: i refund
AI Suggested Response - Confirmation Recommended
Category: REFUND
Intent: track_refund
Suggested Response: I'm on your side your frustration in not being able to see any updates on your refund. Rest assured, I'll do my best to assist you in resolving this issue. To provide you with accurate information, I kindly request your patience as I review the status of your refund. Can you please provide me with the reference number, if available? With this information, I'll be able to investigate further and provide you with the latest updates.
Similarity Score: 0.6388830768535702


In [80]:
workflow_logs.append(workflow_log)
workflow_df=pd.DataFrame(workflow_logs)
workflow_df.head()

,original_query,processed_query,matched_category,matched_intent,similarity_score,decision
0,I need to login into my account,i to login into my account login signin access...,ACCOUNT,registration_problems,0.383953,Escalated to Human Support
1,I need refund,i refund,REFUND,track_refund,0.638883,AI Suggested Response - Confirmation Recommended


# Final Workflow Findings

The workflow simulation demonstrated how an AI-assisted technical support system can automate customer support interactions using TF-IDF based knowledge-base retrieval and confidence-driven decision logic.

The system successfully processed customer queries through a structured workflow involving query preprocessing, similarity-based response retrieval, confidence score evaluation, and escalation decision handling.

High-confidence queries were automatically resolved using knowledge-base responses, while low-confidence queries were flagged for escalation to human support. This workflow reflects realistic operational support environments where AI systems assist with repetitive support requests while human agents manage uncertain or complex issues.

The simulation also highlighted the importance of query preprocessing, text normalization, stopword handling, and query enrichment in improving retrieval accuracy and intent matching performance.

Although the current workflow uses rule-based confidence thresholds and simulated escalation handling, the notebook establishes a strong foundation for future enhancements such as semantic search, embedding-based retrieval, automated ticket routing, real-time dashboard integration, and LLM-assisted response generation.

Overall, the workflow layer complements the retrieval and KPI analytics modules by introducing operational decision-making and support automation capabilities into the AI Technical Support Assistant project.